<a href="https://colab.research.google.com/github/faezesarlakifar/AllerTrans/blob/main/feature-extraction/4.%20ESM-v2-embeddings%20(recombinant).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Embeddings
### We are using the [facebook esm](https://github.com/facebookresearch/esm) model and scripts to extract embedding vectors for recombinant proteins derived from reviewed [UniProt](https://www.uniprot.org/uniprotkb?query=%22recombinant+protein%22&facets=reviewed%3Atrue)  entries. ❤

In [ ]:
# @markdown configs
!git clone https://github.com/facebookresearch/esm.git
!pip install -q git+https://github.com/facebookresearch/esm.git
!pip install -q torch
!pip install -q Bio

Cloning into 'esm'...
remote: Enumerating objects: 1511, done.
remote: Counting objects: 100% (725/725), done.
remote: Compressing objects: 100% (194/194), done.
remote: Total 1511 (delta 567), reused 531 (delta 531), pack-reused 786 (from 1)
Receiving objects: 100% (1511/1511), 12.87 MiB | 24.00 MiB/s, done.
Resolving deltas: 100% (952/952), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# @markdown import necessaries
from google.colab import drive
from tqdm.notebook import tqdm
import esm
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
import shutil
from Bio import SeqIO

In [1]:
# @markdown mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
path = '/content/drive/MyDrive/allergen-detection/fasta-files/'

In [ ]:
# @title Find negative recombinant proteins
from Bio import SeqIO
import random

def read_fasta(file_path):
    return {record.seq: record for record in SeqIO.parse(file_path, "fasta")}

def filter_sequences(file1, file2, output_file, num_to_select=42):
    seqs1 = read_fasta(file1)
    seqs2 = read_fasta(file2)

    # Remove shared sequences
    unique_seqs = {seq: record for seq, record in seqs2.items() if seq not in seqs1}

    # Randomly select 42 sequences
    selected_seqs = random.sample(list(unique_seqs.values()), min(num_to_select, len(unique_seqs)))

    # Write the filtered sequences to a new file
    with open(output_file, "w") as out_f:
        SeqIO.write(selected_seqs, out_f, "fasta")

    print(f"Filtered and saved {len(selected_seqs)} sequences to {output_file}")

In [ ]:
filter_sequences("positive-recombinant.fasta", "all-recombinant.fasta", "negative-recombinant.fasta")

Filtered and saved 42 sequences to negative-recombinant.fasta


In [ ]:
# @title Clean FASTA headers
def clean_fasta_headers(fasta_file, output_file):
    records = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        record.id = record.description.split(" ")[0]
        record.description = record.id
        records.append(record)

    with open(output_file, "w") as out_f:
        SeqIO.write(records, out_f, "fasta")

    print(f"Cleaned FASTA headers and saved to {output_file}")

In [ ]:
clean_fasta_headers("positive-recombinant.fasta", "positive-recombinant-cleaned.fasta")
clean_fasta_headers("negative-recombinant.fasta", "negative-recombinant-cleaned.fasta")

Cleaned FASTA headers and saved to positive-recombinant-cleaned.fasta
Cleaned FASTA headers and saved to negative-recombinant-cleaned.fasta


In [18]:
# @title Save Cleaned Fasta Files
parent_dir = os.path.dirname(path+"positive-recombinant-cleaned.fasta")

os.makedirs(path, exist_ok=True)

fasta_files = ['positive-recombinant-cleaned.fasta', 'negative-recombinant-cleaned.fasta']

for file in fasta_files:
    shutil.copy(file, os.path.join(path, file))
    print(f"Saved {file} to {path}")


Saved positive-recombinant-cleaned.fasta to /content/drive/MyDrive/allergen-detection/fasta-files/
Saved negative-recombinant-cleaned.fasta to /content/drive/MyDrive/allergen-detection/fasta-files/


In [ ]:
# @title Extract embeddings for recombinant positive data
input_file = '/content/positive-recombinant-cleaned.fasta'
!python esm/scripts/extract.py esm2_t33_650M_UR50D $input_file \
embeddings/recombinant_positive --repr_layers 0 32 33 --include mean per_tok

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
Transferred model to GPU
Read /content/positive-recombinant-cleaned.fasta with 42 sequences
Processing 1 of 4 batches (19 sequences)
Processing 2 of 4 batches (11 sequences)
Processing 3 of 4 batches (7 sequences)
Processing 4 of 4 batches (5 sequences)


In [ ]:
# @title Extract embeddings for recombinant negative data
input_file = 'negative-recombinant-cleaned.fasta'
!python esm/scripts/extract.py esm2_t33_650M_UR50D $input_file \
embeddings/recombinant_negative --repr_layers 0 32 33 --include mean per_tok

Transferred model to GPU
Read negative-recombinant-cleaned.fasta with 42 sequences
Processing 1 of 5 batches (16 sequences)
Processing 2 of 5 batches (11 sequences)
Processing 3 of 5 batches (8 sequences)
Processing 4 of 5 batches (4 sequences)
Processing 5 of 5 batches (3 sequences)


In [ ]:
# @title load_embbedings helper function

def load_embbedings(fasta_path, emb_path, allergen_label):
    ys = []
    Xs = []
    missing_files = []

    for header, _seq in esm.data.read_fasta(fasta_path):
        ys.append(allergen_label)
        fn = f'{emb_path}/{header}.pt'

        if os.path.exists(fn):
            try:
                embs = torch.load(fn)
                Xs.append(embs['mean_representations'][33])
            except Exception as e:
                print(f"Error loading {fn}: {e}")
        else:
            missing_files.append(fn)
            print(f"File not found: {fn}. Continuing with other embeddings.")

    if Xs:
        Xs = torch.stack(Xs, dim=0).numpy()
    else:
        print("*")
        Xs = []  # Handle the case when Xs is empty

    # Print the missing files
    if missing_files:
        print("Missing files:")
        for file in missing_files:
            print(file)

    return Xs, ys


## Save the recombinant positive allergen data as a .csv file

In [ ]:
embedding_path = 'embeddings/recombinant_positive'

In [ ]:
fasta_path = 'positive-recombinant-cleaned.fasta'

In [ ]:
Xs, ys = load_embbedings(fasta_path, embedding_path, 1.0)

In [ ]:
len(Xs[0])

1280

In [ ]:
df_recombinant_positive = pd.DataFrame(Xs)
df_recombinant_positive['Label'] = ys
df_recombinant_positive.to_csv('df_recombinant_positive.csv')
df_recombinant_positive.head()

,0,1,2,3,4,5,6,7,8,9,...,1271,1272,1273,1274,1275,1276,1277,1278,1279,Label
0,0.022770,-0.029110,-0.048163,0.085883,-0.056486,-0.099463,0.053803,-0.050036,-0.024854,0.053616,...,0.003424,-0.077201,0.000804,-0.023368,-0.032832,0.057556,-0.068951,-0.064248,0.033488,1.0
1,-0.027281,-0.026025,-0.021719,0.042953,-0.043432,0.057031,0.083801,-0.102275,-0.027197,-0.006529,...,0.107304,0.041301,0.009713,-0.032832,-0.020624,-0.017162,-0.210277,-0.127184,0.138998,1.0
2,0.085734,-0.089933,-0.015315,0.101347,-0.125647,0.010879,0.097136,-0.191397,-0.051163,0.128987,...,-0.089184,-0.104669,0.077777,-0.113041,-0.089945,-0.051328,-0.167994,-0.004427,0.177049,1.0
3,-0.027946,-0.058538,-0.069391,0.090738,0.039675,0.005039,0.004704,-0.202630,-0.025607,0.064304,...,0.003453,0.023845,0.024730,-0.038813,-0.021777,0.049423,-0.196909,-0.053874,0.009511,1.0
4,-0.044899,-0.017458,-0.011260,0.061661,-0.014877,-0.201852,0.110353,-0.058109,-0.067733,0.087626,...,-0.033979,-0.181917,0.055375,-0.012190,-0.067662,0.054758,-0.092389,0.068891,0.001128,1.0


In [ ]:
df_recombinant_positive.to_csv('df_recombinant_positive.csv')

In [ ]:
len(df_recombinant_positive)

42

## Save the recombinant nagative allergen data as a .csv file

In [ ]:
embedding_path = 'embeddings/recombinant_negative'

In [ ]:
fasta_path = 'negative-recombinant-cleaned.fasta'

In [ ]:
Xs, ys = load_embbedings(fasta_path, embedding_path, 0)

In [ ]:
df_recombinant_negative = pd.DataFrame(Xs)
df_recombinant_negative['Label'] = ys
df_recombinant_negative.to_csv('df_recombinant_negative.csv')
df_recombinant_negative.head()

,0,1,2,3,4,5,6,7,8,9,...,1271,1272,1273,1274,1275,1276,1277,1278,1279,Label
0,0.027544,-0.131625,-0.033130,0.034332,-0.031373,-0.046649,0.039784,-0.113883,-0.058210,0.027540,...,-0.037040,0.005336,-0.072119,-0.037879,-0.059686,0.114824,-0.186570,0.003820,0.050671,0
1,0.028553,-0.074669,0.018514,0.068352,0.057935,-0.029153,0.065671,-0.119916,0.029399,0.090551,...,-0.005500,-0.089792,0.055246,-0.006703,-0.107471,0.011454,-0.054124,0.032386,0.031028,0
2,-0.015017,-0.063454,-0.042987,-0.002343,0.001407,0.045996,-0.009629,-0.074849,0.017148,0.045324,...,-0.037468,-0.018136,0.039398,-0.035636,-0.058330,0.080435,-0.117237,-0.049715,0.019538,0
3,-0.019848,-0.018626,0.046368,-0.039652,-0.037423,-0.065299,0.019112,-0.089289,-0.030591,0.035271,...,-0.107178,-0.045060,-0.087152,-0.066959,0.023312,0.092721,-0.160647,0.041998,0.034825,0
4,0.015934,-0.050330,0.097641,0.039054,-0.034239,-0.050415,0.051550,-0.033879,-0.000687,0.021751,...,0.043530,-0.053445,0.055720,0.008639,-0.067399,0.045851,-0.165289,0.031062,-0.003910,0


In [ ]:
df_recombinant_negative.to_csv('df_recombinant_negative.csv')

## Merge train dataframes into a single dataframe

In [ ]:
df_recombinant = pd.concat([df_recombinant_positive, df_recombinant_negative]).sample(frac=1).reset_index(drop=True)

In [ ]:
df_recombinant.head()

,0,1,2,3,4,5,6,7,8,9,...,1271,1272,1273,1274,1275,1276,1277,1278,1279,Label
0,0.010672,-0.057481,-0.100120,0.055157,-0.079812,-0.037094,-0.007852,-0.191705,-0.101635,0.071678,...,-0.024658,0.008858,0.074444,0.020395,0.033964,0.030419,-0.251352,-0.117505,0.104316,0.0
1,0.022770,-0.029110,-0.048163,0.085883,-0.056486,-0.099463,0.053803,-0.050036,-0.024854,0.053616,...,0.003424,-0.077201,0.000804,-0.023368,-0.032832,0.057556,-0.068951,-0.064248,0.033488,1.0
2,-0.015017,-0.063454,-0.042987,-0.002343,0.001407,0.045996,-0.009629,-0.074849,0.017148,0.045324,...,-0.037468,-0.018136,0.039398,-0.035636,-0.058330,0.080435,-0.117237,-0.049715,0.019538,0.0
3,-0.007313,-0.026456,-0.042758,0.072569,-0.037647,-0.103876,0.053726,0.002787,-0.065705,0.109290,...,-0.029615,-0.112200,0.022172,-0.022640,-0.061544,0.108395,-0.167494,-0.056903,-0.019855,0.0
4,0.041665,-0.012955,-0.024071,0.085105,-0.037892,-0.078696,0.026370,-0.020656,-0.009603,0.102361,...,0.008764,-0.081393,0.064784,0.011056,0.044113,0.100494,-0.119706,-0.080247,-0.016273,0.0


In [ ]:
df_recombinant['id'] = df_recombinant.index

In [ ]:
df_recombinant.to_csv('df_recombinant.csv')

In [2]:
file_path = "/content/drive/My Drive/allergen-detection/embeddings/esm-embeddings-with-id/df_recombinant.csv"

In [10]:
parent_dir = os.path.dirname(file_path)
os.makedirs(parent_dir, exist_ok=True)

df_recombinant.to_csv(file_path, index=False)

print(f"CSV file saved to: {file_path}")


CSV file saved to: /content/drive/My Drive/allergen-detection/embeddings/esm-embeddings-with-id/df_recombinant.csv
